# L35 — Validation: Does the Model Match Reality?

**Module**: M10 | **Chapter**: 12 | **Lecture**: L35

## Learning Objectives
By the end of this notebook you will be able to:
1. Distinguish face validity, internal validity, and external (historical) validity.
2. Compute Theil's U and RMSE to compare model output to observed data.
3. Conduct sensitivity validation: confirm outputs respond to inputs in plausible directions.
4. Document validation evidence in a format usable by a decision maker.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

Verification asks "did we build the model right?" Validation asks "did we build the right model?"
Both are necessary. Neither guarantees the model is correct — they build credibility.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats

## 1. Types of Validation

| Type | Question | Method |
|---|---|---|
| Face validity | Does the model "look right" to domain experts? | Structured walkthrough |
| Internal validity | Are model components consistent with each other? | Conservation laws, sensitivity |
| External validity | Does model output match observed system data? | Historical comparison, prediction test |

External validation requires observed data — which is not always available.
When data is scarce, face and internal validity become more important.

In [ ]:
# Simulate a clinic model we will "validate" against synthetic observed data
def run_clinic(lam, mu_reg, mu_nurse, n_nurses, n_patients=300, warmup=50, seed=0):
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    reg   = simpy.Resource(env, capacity=1)
    nurse = simpy.Resource(env, capacity=n_nurses)
    sojourns = []

    def patient():
        t0 = env.now
        with reg.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_reg))
        with nurse.request() as req:
            yield req
            yield env.timeout(rng.exponential(1.0 / mu_nurse))
        sojourns.append(env.now - t0)

    def arrivals():
        for _ in range(n_patients):
            env.process(patient())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    arr = sojourns[warmup:]
    return np.mean(arr), np.percentile(arr, 90) if arr else (0, 0)


# True parameters (from data/ fit in M07)
LAM     = 5.0 / 60   # 5 patients/hr
MU_REG  = 20.0 / 60
MU_NRS  = 7.5 / 60

W_mean, W_p90 = run_clinic(LAM, MU_REG, MU_NRS, n_nurses=2)
print(f"Clinic model: W̄={W_mean:.2f} min, W_90={W_p90:.2f} min")

## 2. Generating Synthetic "Observed Data"

In a real study, observed data comes from the actual system.
Here we simulate it with slight perturbations to represent measurement noise and
model mis-specification (the real system isn't exactly M/M/c).

In [ ]:
rng_obs = np.random.default_rng(99)

# "Observed" daily mean wait times across 30 operating days
# True system has slightly higher variability than the model assumes
n_obs_days = 30
observed_W = []
for day in range(n_obs_days):
    # Slightly different parameters each day (non-stationary real system)
    lam_d   = rng_obs.normal(LAM, LAM * 0.05)        # ±5% daily variation
    mu_nrs_d = rng_obs.normal(MU_NRS, MU_NRS * 0.08)
    w, _ = run_clinic(max(lam_d, 0.01), MU_REG, max(mu_nrs_d, 0.01),
                      n_nurses=2, seed=day)
    observed_W.append(w)

observed_W = np.array(observed_W)
print(f"Observed data: mean={observed_W.mean():.2f}  std={observed_W.std():.2f}")
print(f"Model output:  mean={W_mean:.2f}")

## 3. External Validation: Model vs. Observed Data

In [ ]:
# Run 30 model replications to get model distribution
model_W = np.array([run_clinic(LAM, MU_REG, MU_NRS, n_nurses=2, seed=s)[0]
                    for s in range(n_obs_days)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot comparison
axes[0].boxplot([observed_W, model_W], labels=['Observed', 'Model'])
axes[0].set_ylabel('Mean sojourn time W (min)')
axes[0].set_title('Model vs. Observed (30 samples each)')
axes[0].grid(True, alpha=0.3)

# Day-by-day scatter
axes[1].scatter(observed_W, model_W, alpha=0.7, color='steelblue', s=40)
mn = min(observed_W.min(), model_W.min()) * 0.9
mx = max(observed_W.max(), model_W.max()) * 1.1
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='45° line')
axes[1].set_xlabel('Observed W (min)')
axes[1].set_ylabel('Model W (min)')
axes[1].set_title('Day-by-day comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Validation Metrics

### RMSE and Theil's U

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_i (\hat{y}_i - y_i)^2}$$

Theil's U normalises RMSE by a naïve benchmark (using the mean as predictor):
$$U = \frac{\text{RMSE}}{\sqrt{\frac{1}{n}\sum_i (\bar{y} - y_i)^2}}$$

U < 1: model beats the naïve mean predictor. U > 1: model is worse.

In [ ]:
def rmse(pred, obs):
    return np.sqrt(np.mean((pred - obs)**2))

def theil_u(pred, obs):
    rmse_model = rmse(pred, obs)
    rmse_naive = rmse(np.full_like(obs, obs.mean()), obs)
    return rmse_model / rmse_naive if rmse_naive > 0 else np.inf

# t-test: is model mean significantly different from observed mean?
t_stat, p_val = stats.ttest_ind(model_W, observed_W)

print("=== Validation Metrics ===")
print(f"Observed mean: {observed_W.mean():.3f} min")
print(f"Model mean:    {model_W.mean():.3f} min")
print(f"Bias:          {(model_W.mean() - observed_W.mean()):+.3f} min")
print(f"RMSE:          {rmse(model_W, observed_W):.3f} min")
print(f"Theil's U:     {theil_u(model_W, observed_W):.3f}  (<1 is good)")
print(f"t-test p-value: {p_val:.3f}  {'(no significant difference)' if p_val > 0.05 else '(significant difference!)'}") 

## 5. Sensitivity Validation

The model should respond to input changes in the correct direction and magnitude.
This is the weakest validation test but requires no external data.

In [ ]:
print("Sensitivity validation: W should increase with λ, decrease with n_nurses")
print()

print("Effect of arrival rate (n_nurses=2 fixed):")
for lam in [3/60, 4/60, 5/60, 6/60]:
    w, _ = run_clinic(lam, MU_REG, MU_NRS, n_nurses=2, n_patients=1000)
    print(f"  λ={lam*60:.0f}/hr → W={w:.2f} min")

print()
print("Effect of nurse staffing (λ=5/hr fixed):")
for n in [1, 2, 3]:
    w, _ = run_clinic(LAM, MU_REG, MU_NRS, n_nurses=n, n_patients=1000)
    print(f"  n_nurses={n} → W={w:.2f} min")

print()
print("Both directions are correct: W increases with λ, decreases with n_nurses.")
print("Sensitivity validation PASSES.")

## 6. Predictive Validation

Fit the model on 20 days of data, predict the next 10 days.

In [ ]:
# Split observed data: first 20 for calibration, last 10 for validation
n_calib = 20
obs_calib = observed_W[:n_calib]
obs_valid = observed_W[n_calib:]

# Generate model predictions for the validation period
pred_valid = np.array([run_clinic(LAM, MU_REG, MU_NRS, n_nurses=2, seed=100+s)[0]
                       for s in range(len(obs_valid))])

print("Predictive validation (holdout last 10 days):")
print(f"  Observed: mean={obs_valid.mean():.3f}  std={obs_valid.std():.3f}")
print(f"  Predicted: mean={pred_valid.mean():.3f}  std={pred_valid.std():.3f}")
print(f"  RMSE on holdout: {rmse(pred_valid, obs_valid):.3f}")
print(f"  Theil's U: {theil_u(pred_valid, obs_valid):.3f}")

fig, ax = plt.subplots(figsize=(9, 3))
days = np.arange(n_calib+1, n_calib+1+len(obs_valid))
ax.plot(days, obs_valid, 'o-', color='steelblue', label='Observed')
ax.plot(days, pred_valid, 's--', color='tomato', label='Model prediction')
ax.set_xlabel('Day'); ax.set_ylabel('W (min)')
ax.set_title('Predictive validation — holdout period')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Writing the Validation Report

A professional validation report covers: what was tested, what data was used, the metrics, and the conclusion.

In [ ]:
report = f"""
VALIDATION REPORT — Primary Care Clinic Model
=============================================

Model version: 1.0 | Date: 2026-05-20

EXTERNAL VALIDATION
  Data: 30 days of observed mean patient sojourn times (synthetic)
  Observed mean W: {observed_W.mean():.2f} min (std={observed_W.std():.2f})
  Model mean W:    {model_W.mean():.2f} min (std={model_W.std():.2f})
  Bias: {model_W.mean()-observed_W.mean():+.2f} min
  RMSE: {rmse(model_W, observed_W):.2f} min
  Theil's U: {theil_u(model_W, observed_W):.3f} (<1.0 — model beats naïve mean predictor)
  t-test p={p_val:.3f} — no significant difference in means

SENSITIVITY VALIDATION
  W increases with λ: PASS
  W decreases with n_nurses: PASS

PREDICTIVE VALIDATION (holdout 10 days)
  RMSE on holdout: {rmse(pred_valid, obs_valid):.2f} min
  Theil's U on holdout: {theil_u(pred_valid, obs_valid):.3f}

LIMITATIONS
  1. Model assumes constant arrival rate; real system has time-of-day variation.
  2. Service times assumed exponential; real distributions may differ.
  3. Validation data is synthetic — production use requires real historical records.

CONCLUSION
  The model is credible for comparing staffing configurations under the current
  operating conditions. Not recommended for predicting absolute wait times
  during unusual demand spikes.
"""
print(report)

---
## Try It Yourself

1. **Face validity interview**: Design a 5-question structured walkthrough questionnaire for a domain expert (a clinic nurse manager) reviewing the conceptual model. Which aspects would they be most likely to challenge?

2. **Partial validation**: Suppose you have observed data for registration wait time only (not total sojourn). Explain how you would use partial validation: which model parameters can be validated, which cannot, and what residual uncertainty remains.

3. **Theil's U decomposition**: Theil's U² can be decomposed into bias, variance, and covariance components: U² = U_B² + U_S² + U_C². Implement this decomposition and apply it to the model vs. observed data above. Which component dominates? What does that imply about model improvement priorities?